In [1]:
#> imports
import numpy as np
import scipy as sp
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from scipy.stats import norm
from scipy.stats import truncnorm
from copy import deepcopy
import json
import codecs
from scipy.stats import multivariate_normal
from scipy.stats import invwishart
from scipy.stats import wishart
import pymc as pm

ModuleNotFoundError: No module named 'pymc'

In [ ]:
#> loading paramRanges
fileName = '../paramRanges-2.json'
jsonDump = codecs.open(fileName, 'r', encoding='utf-8').read()
paramRanges = json.loads(jsonDump)
print(paramRanges)

In [ ]:
#> declarations
wmu = []         # mean vector
wcov_tri = []    # triangle of cov matrix, flattened
mean_var = 1e-1  # how different the mean walkers should be
cov_var = 1e-1   # how different the cov walkers should be

#> copying input
ranges = deepcopy(paramRanges)

#> iterating through ranges to create galaxy dict
#> if not varied --> collapse
#> if varied --> store path
#> if dependant on another value --> store path
varied_params, depend_params, names = [], [], {}
for i, profile in enumerate(ranges.keys()):     # each profile (nfw, etc.)
    if profile in ['zl', 'zs', 'pix_arc']: continue # skipping redshifts & conversion
    ranges[profile] = [d for d in ranges[profile] if d['add']] # removing false adds
    for j, dic in enumerate(ranges[profile]):   # each dic in profile array (i.e., [m1, m3, ...]) 
        for k, param in enumerate(dic):         # each param in dic
            if param in ['add', 'm', 'name']:   # params are not fit (skips)
                if param in ['name']: names[dic[param]] =  dic # saves name for dependant params
                continue  # skips
            if not dic[param]['fit']:           # if not fitting
                dic[param] = dic[param]['init'] # sets to init
                if 'p' in str(dic[param]): depend_params.append([param, dic, deepcopy(dic[param])]) # if param depends on another
            else: varied_params.append([profile, j, param, dic[param]]) # if param is varied

#> need to resolve varied params first then dependant params
#> converting to numpy for later
varied_params = np.array(varied_params)
depend_params = np.array(depend_params)

# print(varied_params)
# print(depend_params)
    
#> number of varied parameters
d = len(varied_params) # num dims
freep = ((d)*(d+3))/2  # number of free params searched by mcmc
nWalkers = freep * 2   # number of walkers (determines size of theta0)

#> if no dimensions
if d == 0: error.phrase('PARAMRANGES HAS NO VARIED PARAMS')
    
#> creating mean vector and u&l bounds
mu = np.array([ x['init'] for x in varied_params[:,3] ]) # means (init)
ub = np.array([ x['max']  for x in varied_params[:,3] ]) # upper bound (max)
lb = np.array([ x['min']  for x in varied_params[:,3] ]) # lower bound (min)
rng = np.subtract(ub, lb) # param rang

#> now has gathered all available information from paramRanges

print('mu', mu)
print('ub', ub)
print('lb', lb)
print('rng', rng)

In [ ]:
#> randomizing mean vector
wmu = np.array(mu) + mean_var * np.array(rng) * np.random.randn(len(mu)*2, len(mu))

print(wmu)

#> checking to make sure mean vector values are still within ranges
#> if not --> draw again

#> covariance matrix (identity for now)
cov = np.identity(d)              # base matrix
cov = cov * (rng * cov_var )**2   # setting diags
print(cov)

#> iterating thru covariance matrix
if d > 1:
    for i in range(cov.shape[0]):
        for j in range(cov.shape[1]):
            if i<=j: continue # if diag or triu
            bound = (cov[i][i]*cov[j][j])**0.5
            cov[i][j] = np.random.uniform(-bound, bound)
            

### Prior on variance (or std)

**scale == std** 

**scale == root(var)**

From scipy.stats.truncnorm: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.truncnorm.html

This distribution is the normal distribution centered on *loc* (default 0), with *standard deviation scale* (default 1), and truncated at *a* and *b* standard deviations from *loc*. For arbitrary *loc* and *scale*, *a* and *b* are not the abscissae at which the shifted and scaled distribution is truncated.

**Note**: If *a_trunc* and *b_trunc* are the abscissae at which we wish to truncate the distribution (as opposed to the number of standard deviations from *loc*), then we can calculate the distribution parameters *a* and *b* as follows:

a, b = (a_trunc - loc) / scale, (b_trunc - loc) / scale

This is a common point of confusion. For additional clarification, please see the example below.

In [ ]:
size = 10000
sigma = lambda rng, N: rng / (N)
nmax = 20
nmin = 0.2
min_sigma = sigma(rng, nmin)
max_sigma = sigma(rng, nmax)

print(min_sigma, max_sigma)
print(mu)
cov = np.identity(2)

fig, axes = plt.subplots(1,2,figsize=(12,6))


for i, ax in enumerate(axes.flat):

    print(min_sigma[i])

    ax.axvline(((ub+lb)/2)[i], ls=':', c='k', label='middle')
    ax.axvline(mu[i], ls=':', c='b', label='mean')

    ax.set_xlim(lb[i], ub[i])

    a_min, b_min = (lb[i] - mu[i]) / min_sigma[i], (ub[i] - mu[i]) / min_sigma[i]
    ymin = truncnorm.rvs(loc=mu[i], scale=min_sigma[i], a=a_min, b=b_min, size=size)
    
    a_max, b_max = (lb[i] - mu[i]) / max_sigma[i], (ub[i] - mu[i]) / max_sigma[i]
    ymax = truncnorm.rvs(loc=mu[i], scale=max_sigma[i], a=a_max, b=b_max, size=size)

    ax.hist(ymin, bins=15, label=fr'$\sigma={min_sigma[i]:.1e}; N={(nmin)}\sigma$/rng')
    ax.hist(ymax, bins=15, label=fr'$\sigma={max_sigma[i]:.1e}; N={(nmax)}\sigma$/rng')

    if i == 0:
        ax.set_xlabel('q')
    if i == 1: 
        ax.set_xlabel(r'$\theta$')

    ax.legend()

print(rng/min_sigma)
print(rng/max_sigma)

## Prior on the covariance matrix

The covariance matrix for 2-dimensions is given by:

$$
\Sigma = \begin{bmatrix}
\sigma_1^2 & \sigma_{12} \\
\sigma_{21} & \sigma_2^2 
\end{bmatrix}
$$

Where $\sigma_1^2$ and $\sigma_2^2$ are the variances for parameters $\theta_1$, $\theta_2$ and $\sigma_{12} \equiv \sigma_{21}$ are the covariances. Because the input to MCMC will be $\vec{\theta} = [\overline{\theta_1}, \overline{\theta_2}, \sigma_{1}^2, \sigma_2^2, \sigma_{12}]$, we will need to define priors for the means, variances, and covariances. The means are simple and can be picked from a uniform distribution: $\theta_i$ ~ $U(lb,ub)$, where $ub$=upper bound and $lb$=lower bound (or by a more informative distribution if we so choose). 

Choosing the priors for the variances and covariances are more difficult and non-intuative. One difficulty is that the parameters span different magnitude of values. Therefore, the prior needs to be generalized for all parameters. One way this can be done is to tie the variance $\sigma^2$ to the parameter range $[lb, ub]$. For example, if $q$ spans $[lb=0.5, ub=1.0]$, then the prior should be able to span standard deviation values that makes sense for this range (i.e., $\sigma$=100 would make no sense, $\sigma$=0.1 would). 

One way to do this would be to set upper and lower limits to the number of standard deviations that can fit inside the parameter range. For example, if we wanted no more than 2 standard deviations to fit within the parameter range of $q = [0.5, 1.0]$, then the smallest the standard deviation could be is $\sigma_{q,min} = (1.0-0.5)/2=0.25$. 

In other words, for a Gaussian with $\mu = (ub+lb)/2$, where $ub$=upper bound and $lb$=lower bound, both the $ub$ an $lb$ are $\frac{N}{2} \sigma$ away from the mean $\mu$, where $\sigma = (ub-lb)/N $, where $N$ is the total number of standard deviations within the range $[lb, ub]$. Then, the total number of standard deviations $N$ can be drawn from a uniform distribution that results in a range of reasonable looking Gaussian distributions: $N$ ~ $U(N_{min}, N_{max})$. $N_{min}$ should be low enough to encompass a uniform-looking distribution. $N_{max}$ should be high enough to encompass a steeply peaked Gaussian. 

Below is an example of this with two galaxy parameters axis-ratio $q$ and position-angle $\theta$ with two distributions defined by $N_{min}=0.2$ and $N_{max}=20$. What is slightly confusing is that $N_{min}$ results in the largest standard deviation $\sigma_{max}$, and vice versa.

One additional confusion arises from the implementation of the truncated normal distribution. As input it requires values $a$ and $b$ which describe how many standard deviations $\sigma$ the $lb$ and $ub$ are, respectively, from the mean $\mu$. If the mean falls at the center of the range $\mu=(ub+lb)/2$ then $|a|$=$b$=$N/2$. Luckily there is a simple equation to calculate these $a$ and $b$ values: 

$$
\begin{aligned}
a = (lb - \mu) / \sigma \\\\
b = (up - \mu) / \sigma
\end{aligned}
$$

Positive Semi-Definite Matrix: https://en.wikipedia.org/wiki/Definite_matrix
https://hankyang.seas.harvard.edu/Semidefinite/sdp.html#:~:text=A%20matrix%20A%20is%20said,n%20%C3%97%20n%20PSD%20matrices.

That leaves the covariances. The covariances are given by 

$$
\sigma_{jk} = \frac{1}{n} \sum^n_{i=1} (x_{ij} - \overline{x}_j) (x_{ik} - \overline{x}_k)
$$

and have the requirement that $\sigma_{jk}^2 \le \sigma_{jj} \sigma_{kk}$, where $\sigma_{jj} \equiv \sigma_j^2$. This requirement arises from the fact that the covariance matrix must be a positive semi-definite (PSD) matrix. A $n \times n$ symmetric real matrix $M$ has the requirement $x^T M x \ge 0$ for all $x \in \mathbb{R}^n$ to be PSD. Another way to say this is that a symmetric matrix $M$ is PSD if all of its eigenvalues are non-negative. This results in the requirement: $\sigma_{jk}^2 \le \sigma_{jj} \sigma_{kk}$ (I have a short proof below)


$$
\begin{aligned}
(\sigma^2_i - \lambda)(\sigma^2_j - \lambda) - \sigma^2_{ij} = 0 \\\\
\end{aligned}
$$

$\lambda$ by definition $\ge 0$: if $\lambda = 0 $:
$$
\begin{aligned}
\sigma^2_i \sigma^2_j = \sigma^2_{ij}
\end{aligned}
$$

if $\lambda = 1$:
$$
\begin{aligned}
1 - \sigma^2_i - \sigma^2_j = \sigma^2_{ij} - \sigma^2_i \sigma^2_j
\end{aligned}
$$

if $\sigma^2_{ij} \gt \sigma^2_i \sigma^2_j$, then the left hand side (LHS) can be negative when $\sigma^2_i + \sigma^2_j > 1$ but the RHS cannot be negative! Thus $\sigma^2_{ij} \le \sigma^2_i \sigma^2_j$. This is also known as the covariance inequality which holds via the Cauchy-Schwarz inequality.

Thus, once the variances are determined by drawing $N$'s, then the covariances can be drawn from a uniform distribution $\sigma_{ij}$ ~ $U(-\sqrt{\sigma_i^2 \sigma_j^2}, \sqrt{\sigma_i^2 \sigma_j^2})$.

https://en.wikipedia.org/wiki/Covariance#Relationship_to_inner_products

https://en.wikipedia.org/wiki/Cauchy%E2%80%93Schwarz_inequality

### More positive semi-definite

https://www.mathworks.com/matlabcentral/answers/482818-how-to-define-a-covariance-matrix-that-is-positive-semi-definite

The covariances depend on the other covariances, yippee!!

For a 3D covariance matrix:

$$
\Sigma = \begin{bmatrix}
\sigma_1^2 & \sigma_{12} & \sigma_{13} \\
\sigma_{21} & \sigma_2^2 & \sigma_{23} \\
\sigma_{31} & \sigma_{32} & \sigma_3^2  \\
\end{bmatrix}
$$

Leads to:

$$
\begin{aligned}
(\sigma_i^2 - \lambda)[(\sigma_j^2-\lambda)(\sigma_k^2-\lambda) - \sigma_{jk}^2 ] \\
- \sigma_{ij}[(\sigma_{ij}(\sigma_k^2-\lambda)-\sigma_{ik}\sigma_{jk}] \\
+ \sigma_{ik}[\sigma_{ij}\sigma_{jk} - \sigma_{ik}(\sigma_j^2-\lambda)] = 0
\end{aligned}
$$

If $\lambda=0$:
$$
\begin{aligned}
\sigma_i^2\sigma_j^2\sigma_k^2 - \sigma_i^2\sigma_{jk}^2-\sigma_j^2\sigma_{ik}^2-\sigma_k^2\sigma_{ij}^2+2\sigma_{ij}\sigma_{ik}\sigma_{jk} = 0
\end{aligned}
$$

Solving just for $\sigma_{ik}$:

$$
\sigma_{ik} = \frac{2\sigma_{ij}\sigma_{jk} \pm \sqrt{4\sigma_{ij}^2\sigma_{jk}^2 - 4\sigma_j^2(\sigma_i^2\sigma_{jk}^2+\sigma_k^2\sigma_{ij}^2-\sigma_i^2\sigma_j^2\sigma_k^2)}}{2\sigma_j^2}
$$

In [ ]:
#> third covariance 
def sigma_ik(sigma_i, sigma_j, sigma_k, sigma_ij, sigma_jk):
    
    a = sigma_j**2
    b = - 2 * sigma_ij * sigma_jk
    c = (sigma_i**2 * sigma_jk**2) + (sigma_k**2 * sigma_ij**2) - (sigma_i**2 * sigma_j**2 * sigma_k**2)

    sigma_ikp = (-b + np.sqrt(b**2 - 4*a*c)) / 2*a
    sigma_ikn = (-b - np.sqrt(b**2 - 4*a*c)) / 2*a

    return sigma_ikp, sigma_ikn

In [ ]:
#> num dimensions
d = 3
size = 10000

#> multivariate declarations
mu = np.zeros(shape=d)
cov = np.identity(n=d)
cov = cov*0.1 + 0.9

#> getting third covariance
sigmaik = sigma_ik(cov[0][0], cov[1][1], cov[2][2], cov[0][1], cov[1][2])
print(fr'+ \sigma_ik={sigmaik[0]}, - \sigma_ik={sigmaik[1]}')
cov[0][2] = sigmaik[1]
cov[2][0] = sigmaik[1]

print(cov)
print(f'> The means are {mu}')
print(f'> The covariance matrix is :\n',cov)

#> drawing samples
samples = multivariate_normal.rvs(mean=mu, cov=cov, size=size, random_state=None)
print(samples)

prior matrix

https://en.wikipedia.org/wiki/Wishart_distribution

Σ = D R D

where:

D = diag(σ1,…,σd) (standard deviations)

R = correlation matrix (PSD with ones on the diagonal)

Σij​=σi​σj​Rij​

https://en.wikipedia.org/wiki/Lewandowski-Kurowicka-Joe_distribution

Standard deviation–correlation decomposition of a covariance matrix (the Barnard–McCulloch–Meng separation strategy).

Cholesky parameterization

(L @ L.T) is actually using a Cholesky parameterization, which is a different but related PSD construction method. Both approaches are widely used depending on whether you want interpretable variances + correlations or efficient unconstrained parameters.

In [ ]:
def sample_covariance_DRD(d):

    # sample std deviations
    sigma = np.exp(np.random.normal(0,1,size=d))

    # S = invwishart.rvs(df=d+1, scale=np.identity(d))
    S = wishart.rvs(df=d, scale=np.identity(d))

    #> convert to correlation matrix
    Dcorr = np.sqrt(np.diag(S)) # .reshape(d,1) # needed if wanting to use Dcorr @ Dcorr.T
    R = S / np.outer(Dcorr, Dcorr) # why not Dcorr @ Dcorr.T? (it is the same!)

    # construct covariance
    D = np.diag(sigma)
    Sigma = D @ R @ D

    return Sigma

d=12
mu = np.exp(np.random.normal(0,1,size=d))
for i in range(10000):
    cov = sample_covariance_DRD(d=d)
    samples = multivariate_normal.rvs(mean=mu, cov=cov, size=1, random_state=None)
print(cov)

In [ ]:
size = 100000
u1 = np.random.uniform(low=0, high=1, size=size)
u2 = np.random.uniform(low=0, high=1, size=size)
u3 = u1 * u2

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(19,6))

for ax, u in zip(axes, [u1, u2, u3]):
    ax.hist(u)

In [ ]:
def ranCorr(d):
    
    #> random wishart covariance?
    # https://www.math.wustl.edu/~sawyer/hmhandouts/Wishart.pdf
    #A = np.random.normal(size=(d,d))
    #S = A @ A.T

    S = invwishart.rvs(df=d+1, scale=np.identity(d))
    # S = wishart.rvs(df=d+2, scale=np.identity(d))

    #> convert to correlation matrix
    Dcorr = np.sqrt(np.diag(S)) # .reshape(d,1) # needed if wanting to use Dcorr @ Dcorr.T
    R = S / np.outer(Dcorr, Dcorr) # why not Dcorr @ Dcorr.T? (it is the same!)
    return R

In [ ]:
#> plotting how the prior on the correlation matrix changes with respect to dimension size

#> initializing plot
fig, ax = plt.subplots(1,1,figsize=(6,6))

ax.grid(ls=':', alpha=0.3)
ax.set_xlabel(r'$r_{12}$')
ax.set_ylabel('Count')
ax.set_xlim(-1, 1)

for i in range(1,7):

    d = i*2 + 1
    r12 = []
    for j in range(10000):
        r12.append(ranCorr(d)[0][2])

    ax.hist(r12, density=True, histtype='step', label=f'd={d}')

ax.legend()

plt.show()

https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wishart.html 

https://en.wikipedia.org/wiki/Wishart_distribution

In [ ]:
#import pymc
#sd_dist = pymc.Uniform(name='std', rng=[0,1])
#print(sd_dist)
#pymc.LKJCholeskyCov(name='name', eta=1, n=d, sd_dist=sd_dist)

from tqdm import tqdm

In [ ]:
ub = 10
lb = -10
d=2

samples = []
for i in tqdm(range(100)):
    #> drawing mean vector and standard deviations
    mu_model = pm.Uniform.dist(lower=lb, upper=ub, shape=d)
    sigma_model = pm.Uniform.dist(lower=(ub-lb)/1e3, upper=(ub-lb)/1e1, size=d)
    
    #> LKJ random correlation matrix R
    eta = 1.0     # uniform
    n = d         # num dims
    cov_model = pm.LKJCholeskyCov.dist(eta=eta, n=n, sd_dist=sigma_model)
    
    mu = pm.draw(mu_model)
    chol, corr, sigmas = pm.draw(cov_model)
    cov = chol @ chol.T # equiv to cov = np.diag(sigmas) @ corr @ np.diag(sigmas)
    samples.append(cov[0][0])

fig, ax = plt.subplots(1,1,figsize=(6,6))
ax.hist(samples)
plt.show()

In [ ]:

samples = []
stds = []
for i in tqdm(range(10000)):
    # sample std deviations
    mu = np.random.uniform(lb,ub,size=d)
    std = np.random.uniform((ub-lb)/1e3,(ub-lb)/1e1,size=d)
    std = [1, 1]
    # lower=(ub-lb)/1e3, upper=(ub-lb)/1e1
    #> random wishart covariance?
    # https://www.math.wustl.edu/~sawyer/hmhandouts/Wishart.pdf
    #A = np.random.normal(size=(d,d))
    #S = A @ A.T

    S = invwishart.rvs(df=d+1, scale=np.identity(d))
    # S = wishart.rvs(df=d, scale=np.identity(d))

    #> convert to correlation matrix
    Dcorr = np.sqrt(np.diag(S)) # .reshape(d,1) # needed if wanting to use Dcorr @ Dcorr.T
    R = S / np.outer(Dcorr, Dcorr) # why not Dcorr @ Dcorr.T? (it is the same!)
    
    Sigma = np.diag(std) @ R @ np.diag(std)
    samples.append(Sigma[0][1])
    stds.append(std[0])
    
fig, ax = plt.subplots(1,1,figsize=(6,6))
ax.hist(samples)
# ax.hist(stds)
plt.show()